In [11]:
import os, time, pandas as pd
from dataclasses import dataclass, asdict
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- config --------------------
CNEL_URL = "https://www.cnel.it/Archivio-Contratti-Collettivi/Archivio-Nazionale-dei-contratti-e-degli-accordi-collettivi-di-lavoro/Contrattazione-Nazionale/Ricerca-CCNL"

def make_driver():
    opts = Options()
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1920,1080")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

@dataclass
class AgreementItaly:
    titolo: str; tipologia: str; codice_cnel: str; data_stipula: str; pdf_link: str | None

def scrape_italy_final_attempt(out_dir="italy_data", target_count=50):
    abs_out_dir = os.path.abspath(out_dir)
    os.makedirs(abs_out_dir, exist_ok=True)
    
    d = make_driver()
    results = []

    try:
        print(f"Loading {CNEL_URL}...")
        d.get(CNEL_URL)
        
        print("Attempting to trigger search automatically...")
        try:
            # Try a generic selector for any submit button in the search area
            search_btn = d.find_element(By.CSS_SELECTOR, "input[type='submit'], .btn-primary, button[id*='submit']")
            d.execute_script("arguments[0].click();", search_btn)
        except:
            print("Auto-click failed. If the table isn't showing, please click 'Cerca' manually.")

        # --- THE SENTRY ---
        print("Waiting for results table to appear (Timeout in 60s)...")
        # We wait for the specific table class seen in your screenshots
        WebDriverWait(d, 60).until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.views-table tbody tr")))
        
        print("Table detected! Scraping now...")
        time.sleep(2) 

        rows = d.find_elements(By.CSS_SELECTOR, "table.views-table tbody tr")
        
        for i, row in enumerate(rows[:target_count]):
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) < 5: continue
            
            # Mapping from your 8-column screenshot:
            # [1] Titolo | [2] Tipologia | [3] Codice | [4] Data Stipula
            titolo = cols[1].text.strip()
            tipologia = cols[2].text.strip()
            codice = cols[3].text.strip()
            data = cols[4].text.strip()
            
            try:
                # Last column has the download icon link
                pdf_link = cols[-1].find_element(By.TAG_NAME, "a").get_attribute("href")
            except:
                pdf_link = "No PDF"

            ag = AgreementItaly(
                titolo=titolo, tipologia=tipologia, 
                codice_cnel=codice, data_stipula=data, 
                pdf_link=pdf_link
            )
            results.append(ag)
            print(f"   [{i+1}] {codice} saved.")

    finally:
        d.quit()
        if results:
            df = pd.DataFrame([asdict(r) for r in results])
            csv_path = os.path.join(abs_out_dir, "italy_final_links.csv")
            # utf-8-sig ensures Italian accents work in Excel
            df.to_csv(csv_path, index=False, encoding="utf-8-sig")
            print(f"\nSUCCESS! {len(results)} links saved to {csv_path}")
            return df

# RUN
scrape_italy_final_attempt(target_count=50)

Loading https://www.cnel.it/Archivio-Contratti-Collettivi/Archivio-Nazionale-dei-contratti-e-degli-accordi-collettivi-di-lavoro/Contrattazione-Nazionale/Ricerca-CCNL...
Attempting to trigger search automatically...
Waiting for results table to appear (Timeout in 60s)...


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=146.0.7680.81)
Stacktrace:
0   chromedriver                        0x0000000100692a80 cxxbridge1$str$ptr + 3158700
1   chromedriver                        0x000000010068ab08 cxxbridge1$str$ptr + 3126068
2   chromedriver                        0x000000010015f724 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75036
3   chromedriver                        0x0000000100138200 chromedriver + 164352
4   chromedriver                        0x00000001001d00b0 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 536232
5   chromedriver                        0x00000001001e5e20 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 625688
6   chromedriver                        0x000000010019cfc8 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 327104
7   chromedriver                        0